# Algorithmic Methods of Data Mining and Laboratory

# Lab Session 05 - Clustering

In [ ]:
import random
import numpy as np
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
import matplotlib as mpl

## Partitional Clustering with $K$-Means

Given a dataset $\mathcal{D} = \{\mathbf{x}_i\}_{i = 1}^n$, the $K$-means optimization problem aims to find a $K$-partition $C_1, \dots, C_K \subseteq \mathcal{D}$ and a sequence of representatives $\mathbf{c}_1, \dots, \mathbf{c}_K \in \mathbb{R}^d$ that allow to minimize the cost function
$$\mathcal{L} = \sum_{j = 1}^K\sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \mathbf{c}_j \rVert_2^2$$

In [ ]:
# Create 1500 random samples, which are grouped in 5 clusters, and plot them.
n_samples = 1500
random_state = 1700
X, y = make_blobs(n_samples=n_samples, random_state=random_state, centers=5)
plt.scatter(X[:, 0], X[:, 1], c=y)
plt.show()

In [ ]:
# Define the number of clusters and the number of points to generate within a certain deviation from each centroid.
num_clusters = 5
num_points = 1000
stdeviation = 1.

# Create the centroids.
centers = []
for i in range(num_clusters):
  # Generate centroids from a Unif(-10, 10) distribution.
  center_x = np.random.uniform(-10, 10)
  center_y = np.random.uniform(-10, 10) # random.random().
  centers.append((center_x,center_y))
centers = np.array(centers)
centers

In [ ]:
# Plot the centroids.
plt.scatter(centers[:,0],centers[:,1])
plt.show()

In [ ]:
points = []
for i in range(num_clusters):
  for j in range(num_points // num_clusters):
    # Generate data points to create a cluster around each centroid.
    point_x = np.random.normal(centers[i, 0], stdeviation)
    point_y = np.random.normal(centers[i, 1], stdeviation)
    points.append((point_x, point_y))
points = np.array(points)

In [ ]:
# Plot the resulting clusters.
plt.scatter(points[:, 0], points[:, 1])
plt.scatter(centers[:, 0], centers[:, 1])
plt.show()

One of the main problems of the $K$-means problem lies in the fact that finding an optimal solution is a $\mathbf{NP}$-hard problem.

For this reason, this problem is typically tackled using $(1 + \varepsilon)$-approximation algorithms that run in $O(n^{\frac{1}{\varepsilon}})$ time.

### Lloyd's Method

Lloyd's method is the main approximation algorithm used for the $K$-means problem.

This algorithm starts with an initialization phase, where the $K$ representatives are chosen at random, followed by an iterative phase that alternates between an assignment step, where each data point is mapped to the nearest cluster, and an update step, where each representative is calibrated to minimize the cost function, going on until convergence.

While convergence is typically considered to be reached whenever the cost function stops changing significantly across iterations, it is also possible to fix a maximum number of iterations before the algorithm is forced to terminate.

In [ ]:
# Manual implementation of the K-Means problem using Lloyd's method.

# Start by choosing the number of clusters and the number of iterations to perform before stopping.
k = 5
num_iterations = 10 # Use the iteration limit as a stopping criterion.
plot_every_iteration = True
np.random.seed(210) # For reproducibility.

# Initialize the centroids using k random data points from the dataset.
idxs = random.sample([i for i in range(X.shape[0])], size=k, replace=False) # Choose without replacement.
centroids = X[idxs]

for _ in range(num_iterations):
  old_centroids = centroids # Keep track of the previous centroids for convergence.

  # Assignment Step: Compute the distances between data points and centroids.
  #                  Next, use these distances to map each data point to the nearest cluster.
  distances = np.zeros((X.shape[0], k)) # This is a (1500 x 5) matrix.

  for i in range(X.shape[0]):
    # Iterate through all combinations of data points and centroids and store their distances in the matrix.
    xi = X[i,]
    for j in range(k):
      cj = centroids[j]
      distances[i, j] = np.linalg.norm(xi - cj) # Use the Euclidean norm as a distance function.

  cluster_per_point = np.argmin(distances, axis=1) # Find the nearest cluster for each data point.

  # If needed, it is possible to look at the current clusters.
  if plot_every_iteration:
    plt.scatter(X[:, 0], X[:, 1], c=cluster_per_point)
    plt.scatter(centroids[:,0], centroids[:,1], c='red', s=200, marker='X')
    plt.show()

  # Update Step: Calibrate each centroid to minimize the cost function.
  for i in range(k):
    # Find the points mapped to cluster i and update the corresponding centroid to their average.
    idxs_k = np.array([r for r in range(X.shape[0]) if cluster_per_point(r) == i])
    points_per_cluster_k = X[idxs_k]
    centroids[i] = np.mean(points_per_cluster_k, axis=0)

  # Define a stopping criterion for convergence.
  distances = [np.linalg.norm(old_centroids[i] - centroids[i]) for i in range(k)]
  if max(distances) < 0.001:
    # Terminate the algorithm when the maximum cluster distance does not exceed a chosen threshold.
    break

In [ ]:
# Professor's implementation of the K-Means problem.

def initialize_centroids(X, k):
  n_samples = X.shape[0]
  random_indices = np.random.choice(n_samples, k, replace=False)
  centroids = X[random_indices]
  return centroids

def assign_clusters(X, centroids):
  distances = np.linalg.norm(X[:, np.newaxis] - centroids, axis=2)
  cluster_labels = np.argmin(distances, axis=1)
  return cluster_labels

def update_centroids(X, cluster_labels, k):
  new_centroids = np.array([X[cluster_labels == i].mean(axis=0) for i in range(k)])
  return new_centroids

def k_means(X, k, max_iters=100, plot_every_iteration=False, tol=1e-4):
  # Initialize centroids.
  centroids = initialize_centroids(X, k)

  for i in range(max_iters):
    # Assign clusters and update centroids accordingly.
    cluster_labels = assign_clusters(X, centroids)
    new_centroids = update_centroids(X, cluster_labels, k)

    if plot_every_iteration:
      plt.scatter(X[:, 0], X[:, 1], c=cluster_labels)
      plt.scatter(new_centroids[:, 0], new_centroids[:, 1], c='red', s=200, marker='X')
      plt.show()

    # Check for convergence by checking whether cluster updates are significant or not.
    if np.linalg.norm(new_centroids - centroids) < tol:
      break

    centroids = new_centroids

  return centroids, cluster_labels

In [ ]:
# Run the K-Means problem.
k = 4
centroids, cluster_labels = k_means(X, k, plot_every_iteration = True)

# Plot the results
plt.scatter(X[:, 0], X[:, 1], c=cluster_labels, cmap='viridis', s=30, label='Data Points')
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', s=200, marker='X', label='Centroids')
plt.title("K-Means Clustering")
plt.legend()
plt.show()

## Clustering on Transformed Data

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans

In [ ]:
n_samples = 1500
random_state = 170
X, y = make_blobs(n_samples=n_samples, random_state=random_state, centers=3)
plt.scatter(X[:, 0], X[:, 1], c=y)
plt.show()

It is possible to transform a collection of data points $\mathbf{x}_1, \dots, \mathbf{x}_n \in \mathbb{R}^2$ using a matrix $\mathbf{A} \in \mathbb{R}^{2 \times 2}$ in which the first row indicates how the $x$-coordinates are transformed while the second row indicates how the $y$-coordinates are transformed.

In [ ]:
# Create anisotropic blobs where each component has different variance, resulting in stretched clusters.
transformation = [[0.60834549, -0.63667341], [-0.40887718, 0.85253229]]
X_aniso = np.dot(X, transformation)

# Plot the resulting transformation.
plt.scatter(X_aniso[:, 0], X_aniso[:, 1])
plt.show()

In [ ]:
# Create blobs with different standard deviations, resulting in very small borders between clusters.
X_varied, y_varied = make_blobs(n_samples=n_samples, cluster_std=[1.0, 2.5, 0.5], random_state=random_state)

# Plot the resulting transformation.
plt.scatter(X_varied[:, 0], X_varied[:, 1])
plt.show()

In [ ]:
# Create unevenly sized blobs.
X_filtered = np.vstack((X[y == 0][:500], X[y == 1][:100], X[y == 2][:10]))
y_filtered = [0] * 500 + [1] * 100 + [2] * 10

# Plot the resulting transformation.
plt.scatter(X_filtered[:, 0], X_filtered[:, 1])
plt.show()

In [ ]:
# Visually compare each clustering transformation.
fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(12, 12))

axs[0, 0].scatter(X[:, 0], X[:, 1], c=y)
axs[0, 0].set_title("Mixture of Gaussian Blobs")

axs[0, 1].scatter(X_aniso[:, 0], X_aniso[:, 1], c=y)
axs[0, 1].set_title("Anisotropically Distributed Blobs")

axs[1, 0].scatter(X_varied[:, 0], X_varied[:, 1], c=y_varied)
axs[1, 0].set_title("Unequal Variance")

axs[1, 1].scatter(X_filtered[:, 0], X_filtered[:, 1], c=y_filtered)
axs[1, 1].set_title("Unevenly Sized Blobs")

plt.suptitle("Ground Truth Clusters").set_y(0.95)
plt.show()

## Hierarchical Clustering

In [ ]:
from scipy.cluster.hierarchy import dendrogram
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import fcluster
# from sklearn.datasets import load_iris

Hierarchical clustering algorithms are used to partition a dataset into a tree-like hierarchy that can be constructed using top-down divisive methods or bottom-up agglomerative methods.

In [ ]:
def plot_dendrogram(model, **kwargs):
  # Create a linkage matrix and then plot the corresponding dendrogram.
  counts = np.zeros(model.children_.shape[0])
  n_samples = len(model.labels_)
  for i, merge in enumerate(model.children_):
    current_count = 0
    for child_idx in merge:
      if child_idx < n_samples:
        current_count += 1
      else:
        current_count += counts[child_idx - n_samples]
    counts[i] = current_count

  linkage_matrix = np.column_stack([model.children_, model.distances_, counts]).astype(float)

  # Plot the resulting dendrogram.
  dendrogram(linkage_matrix, **kwargs)
  return linkage_matrix


# iris = load_iris()
# X = iris.data

# Set distance_threshold = 0 to compute the full tree.
model = AgglomerativeClustering(distance_threshold=0, n_clusters=None)

model = model.fit(X)
plt.title("Hierarchical Clustering Dendrogram")

# Plot the top three levels of the dendrogram.
linkage_matrix = plot_dendrogram(model, truncate_mode="level", p=5)
plt.xlabel("Points in Node")
plt.show()

In [ ]:
# Choose a maximum distance to cut the dendrogram.
max_distance = 20
labels = fcluster(linkage_matrix, max_distance, criterion='distance')
print(labels)

## The Elbow Method

In [ ]:
from sklearn.cluster import KMeans
from sklearn.datasets import make_blobs
from yellowbrick.cluster import KElbowVisualizer

When performing $K$-means, the elbow method is a heuristic rule that allows to find a suitable number of clusters depending on a chosen evaluation metric.

The distortion score of a $K$-means solution assesses clustering quality in terms of the sum of squared distances between each data point and the center of the assigned clustering.
$$WCSS = \sum_{j = 1}^K\sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \mathbf{c}_j \rVert_2^2$$

In [ ]:
# Use the elbow method on a simple blob.
X, y = make_blobs(n_samples=1000, n_features=12, centers=8, random_state=42)

# Instantiate the clustering model and visualizer.
model = KMeans()
visualizer = KElbowVisualizer(model, k=(4,12), timings=False)
visualizer.fit(X) # Fit the data to the visualizer.
visualizer.show() # Finalize and render the figure.

The Calinski-Harabasz score measures clustering quality in terms of the ratio of between-cluster dispersion to within-cluster dispersion.
$$CH = \frac{BCSS}{WCSS} \cdot \frac{n - K}{K - 1} = \frac{\sum_{j = 1}^K |C_j|\lVert \mathbf{c}_j - \overline{\mathbf{c}} \lVert_2^2}{\sum_{j = 1}^K\sum_{\mathbf{x}_i \in C_j} \lVert \mathbf{x}_i - \mathbf{c}_j \rVert_2^2} \cdot \frac{n - K}{K - 1}$$

In [ ]:
model = KMeans()
visualizer = KElbowVisualizer(model, k=(4,12), metric='calinski_harabasz', timings=False)
visualizer.fit(X) # Fit the data to the visualizer.
visualizer.show() # Finalize and render the figure.

The silhouette score evaluates clustering quality by measuring how similar each data point $\mathbf{x}_i$ is to the points in its assigned cluster $a$ compared to the data points of the nearest cluster $b$ it does not belong to.
$$S(\mathbf{x}_i) = \frac{b(\mathbf{x}_i) - a(\mathbf{x}_i)}{\max(a(\mathbf{x}_i), b(\mathbf{x}_i))} \text{ } \forall \mathbf{x}_i \in \mathcal{D}$$

In [ ]:
model = KMeans()
visualizer = KElbowVisualizer(
model, k=(4,12), metric='silhouette', timings=False)
visualizer.fit(X) # Fit the data to the visualizer.
visualizer.show() # Finalize and render the figure.